# Maze Decision Transformer

## 1 · Imports & Hyperparameters

In [2]:
import os
import pickle
import time
import math
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} | device count: {torch.cuda.device_count()}')
print(f'AMP      : {torch.cuda.is_available()}')

PyTorch  : 2.5.1+cu124
CUDA     : True | device count: 1
AMP      : True


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          ALL HYPERPARAMETERS LIVE HERE — EDIT FREELY        ║
# ╚══════════════════════════════════════════════════════════════╝
HP: Dict = {
    # ── Maze ────────────────────────────────────────────────────
    'maze_rows'             : 200,
    'maze_cols'             : 200,
    'extra_opening_prob'    : 0.25,
    'seed'                  : 42,

    # ── Data collection ─────────────────────────────────────────
    'n_per_eps'             : 100,
    'epsilons'              : [0.0, 0.2, 0.4, 0.6, 0.8],
    'max_steps'             : 3000,
    'max_attempts_multiplier': 50,

    # ── Dataset ─────────────────────────────────────────────────
    'context_len'           : 150,
    'n_samples'             : 100_000,
    'val_fraction'          : 0.1,   # fraction held out for validation

    # ── Model ───────────────────────────────────────────────────
    'hidden_dim'            : 256,
    'n_heads'               : 4,
    'n_layers'              : 3,
    'dropout'               : 0.1,

    # ── Training ────────────────────────────────────────────────
    'batch_size'            : 256,
    'epochs'                : 30,
    'lr'                    : 6e-4,
    'weight_decay'          : 1e-4,
    'warmup_steps'          : 1000,
    'grad_clip'             : 0.25,
    'grad_accum_steps'      : 1,     # increase to simulate larger batch
    'use_amp'               : True,  # mixed-precision (CUDA only)
    'use_compile'           : False, # torch.compile (PyTorch >= 2.0)

    # ── Early stopping ──────────────────────────────────────────
    'patience'              : 5,     # epochs without val-loss improvement
    'min_delta'             : 1e-4,  # min improvement to reset counter

    # ── Evaluation ──────────────────────────────────────────────
    'eval_rollouts'         : 20,
    'rtg_delta'             : 12.0,   # |target_rtg - net_return| <= delta → SUCCESS

    # ── I/O ─────────────────────────────────────────────────────
    'traj_file'             : 'trajectories.pkl',
    'ckpt_file'             : 'dt_checkpoint_testing.pt',
    'eval_file'             : 'alignment_results.pkl',
}

# ── Derived constants ────────────────────────────────────────────
N_ACTIONS = 4   # 0=up 1=down 2=left 3=right
N_STATES  = HP['maze_rows'] * HP['maze_cols']
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Global seeding ───────────────────────────────────────────────
def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(HP['seed'])
print(f'Device: {DEVICE}  |  N_STATES: {N_STATES}  |  N_ACTIONS: {N_ACTIONS}')

Device: cuda  |  N_STATES: 40000  |  N_ACTIONS: 4


## 2 · Maze Generation & Environment

In [4]:
def make_maze(
    rows: int,
    cols: int,
    seed: int = 42,
    extra_opening_prob: float = 0.25,
) -> np.ndarray:
    """
    Random perfect maze via iterative DFS, with optional extra openings for cycles.

    Physical grid shape: (2*rows+1) x (2*cols+1)
    Value 1 = wall, 0 = open passage.
    """
    rng = np.random.RandomState(seed)
    h, w = 2 * rows + 1, 2 * cols + 1
    grid    = np.ones((h, w), dtype=np.uint8)
    visited = np.zeros((rows, cols), dtype=bool)
    DIRS    = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    stack = [(0, 0)]
    visited[0, 0] = True
    grid[1, 1]    = 0

    while stack:
        r, c = stack[-1]
        nbrs = [
            (r + dr, c + dc, dr, dc)
            for dr, dc in DIRS
            if 0 <= r + dr < rows and 0 <= c + dc < cols
            and not visited[r + dr, c + dc]
        ]
        if nbrs:
            nr, nc, dr, dc = nbrs[rng.randint(len(nbrs))]
            visited[nr, nc] = True
            grid[2*r+1+dr, 2*c+1+dc] = 0
            grid[2*nr+1,   2*nc+1  ] = 0
            stack.append((nr, nc))
        else:
            stack.pop()

    # Random extra openings (cycles)
    if extra_opening_prob > 0:
        for r in range(rows):
            for c in range(cols):
                for dr, dc in ((0, 1), (1, 0)):
                    nr, nc = r + dr, c + dc
                    if nr >= rows or nc >= cols:
                        continue
                    wr, wc = 2*r+1+dr, 2*c+1+dc
                    if grid[wr, wc] == 1 and rng.rand() < extra_opening_prob:
                        grid[wr, wc] = 0
    return grid

In [5]:
class MazeEnv:
    """
    Discrete maze environment.

    State  : flat cell index  row*cols + col
    Actions: 0=up 1=down 2=left 3=right
    Reward : -1 per step
    Done   : reaching (rows-1, cols-1)
    """
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def __init__(self, grid: np.ndarray, rows: int, cols: int):
        self.grid = grid
        self.rows = rows
        self.cols = cols
        self.start = (0, 0)
        self.goal  = (rows - 1, cols - 1)
        self.pos   = self.start

    def enc(self, pos) -> int:
        return pos[0] * self.cols + pos[1]

    def reset(self) -> int:
        self.pos = self.start
        return self.enc(self.pos)

    def valid_actions(self, pos=None) -> List[int]:
        if pos is None:
            pos = self.pos
        r, c = pos
        out = []
        for a, (dr, dc) in enumerate(self.DIRS):
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.rows and 0 <= nc < self.cols:
                if self.grid[2*r+1+dr, 2*c+1+dc] == 0:
                    out.append(a)
        return out

    def step(self, action: int) -> Tuple[int, float, bool]:
        r, c = self.pos
        dr, dc = self.DIRS[action]
        nr, nc = r + dr, c + dc
        if (
            0 <= nr < self.rows and 0 <= nc < self.cols
            and self.grid[2*r+1+dr, 2*c+1+dc] == 0
        ):
            self.pos = (nr, nc)
        done   = self.pos == self.goal
        reward = -1.0
        return self.enc(self.pos), reward, done

In [6]:
def visualise_maze(grid: np.ndarray, env: MazeEnv, filename: str = 'maze.png') -> None:
    """Save a colour PNG of the maze (black=wall, white=passage, green=start, red=goal)."""
    h, w  = grid.shape
    img   = np.ones((h, w, 3), dtype=np.float32)
    walls = grid == 1
    img[walls] = [0, 0, 0]

    # Light-blue cell centres
    img[1::2, 1::2] = [0.8, 0.9, 1.0]

    sr, sc = env.start
    img[2*sr+1, 2*sc+1] = [0.0, 0.85, 0.0]   # green start

    gr, gc = env.goal
    img[2*gr+1, 2*gc+1] = [0.9, 0.1, 0.1]    # red goal

    fig, ax = plt.subplots(figsize=(7, 7), dpi=120)
    ax.imshow(img, interpolation='nearest')
    ax.axis('off')
    patches = [
        mpatches.Patch(color=[0, 0.85, 0], label='Start'),
        mpatches.Patch(color=[0.9, 0.1, 0.1], label='Goal'),
    ]
    ax.legend(handles=patches, loc='upper right', fontsize=8)
    ax.set_title(f'Maze {env.rows}×{env.cols}', fontsize=10)
    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight', pad_inches=0)
    plt.show()
    plt.close()
    print(f'Saved maze image → {filename}')

## 3 · BFS Helper & Trajectory Collection

In [7]:
def bfs_dist_to_goal(env: MazeEnv) -> Dict:
    """BFS from goal backwards to get shortest-path distance for every cell."""
    dist = {env.goal: 0}
    q    = deque([env.goal])
    while q:
        p  = q.popleft()
        d0 = dist[p]
        for a in env.valid_actions(p):
            r, c   = p
            dr, dc = env.DIRS[a]
            npos   = (r + dr, c + dc)
            if npos not in dist:
                dist[npos] = d0 + 1
                q.append(npos)
    return dist


def greedy_bfs_action(env: MazeEnv, pos, dist: Dict) -> Optional[int]:
    best_a, best_d = None, float('inf')
    for a in env.valid_actions(pos):
        r, c   = pos
        dr, dc = env.DIRS[a]
        d      = dist.get((r + dr, c + dc), float('inf'))
        if d < best_d:
            best_d, best_a = d, a
    return best_a

In [8]:
def collect_trajectories(
    env: MazeEnv,
    dist_to_goal: Dict,
    n_traj: int,
    epsilon: float,
    max_steps: int,
    seed: int,
    max_attempts_multiplier: int = 50,
) -> Tuple[List[Dict], int]:
    """
    Collect `n_traj` successful (goal-reaching) episodes.

    Policy: epsilon-greedy BFS.
      epsilon=0 → near-optimal  |  epsilon=1 → fully random

    Returns (list-of-trajectories, total-attempts).
    Each trajectory dict: states, actions, rewards, total_reward, reached_goal.
    """
    rng      = np.random.RandomState(seed)
    trajs    = []
    attempts = 0
    max_att  = max(n_traj, n_traj * max_attempts_multiplier)

    while len(trajs) < n_traj and attempts < max_att:
        attempts += 1
        env.reset()
        pos     = env.start
        states, actions, rewards = [], [], []

        for _ in range(max_steps):
            states.append(env.enc(pos))
            valid = env.valid_actions(pos)
            if not valid:
                break

            greedy_a = greedy_bfs_action(env, pos, dist_to_goal)
            a = int(rng.choice(valid)) if (rng.rand() < epsilon or greedy_a is None) else int(greedy_a)

            actions.append(a)
            _, r, done = env.step(a)
            rewards.append(r)
            pos = env.pos
            if done:
                break

        if actions and pos == env.goal:
            rew_arr = np.array(rewards, dtype=np.float32)
            trajs.append({
                'states'      : np.array(states,  dtype=np.int32),
                'actions'     : np.array(actions, dtype=np.int32),
                'rewards'     : rew_arr,
                'total_reward': float(rew_arr.sum()),
                'reached_goal': True,
            })

    return trajs, attempts

## 4 · Dataset

In [9]:
class MazeTrajectoryDataset(Dataset):
    """
    Random context windows drawn from offline trajectories.

    Each sample: (states, actions, rtgs, timesteps, mask) of length context_len.
    Short trajectories are right-padded; long ones are randomly cropped.
    Trajectories are sampled proportionally to their length.
    """

    def __init__(
        self,
        trajectories: List[Dict],
        context_len: int,
        rtg_scale: float,
        n_samples: int,
    ):
        self.K         = context_len
        self.rtg_scale = float(rtg_scale)
        self.n_samples = n_samples

        # Filter & precompute RTGs
        self.trajs = []
        for t in trajectories:
            if not t.get('reached_goal', False) or len(t['actions']) == 0:
                continue
            rtg = np.cumsum(t['rewards'][::-1])[::-1].astype(np.float32)
            self.trajs.append({**t, 'rtg': rtg})

        if not self.trajs:
            raise ValueError('No successful trajectories in dataset.')

        lens          = np.array([len(t['actions']) for t in self.trajs], dtype=np.float64)
        self.sample_p = lens / lens.sum()

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, _):
        t = self.trajs[np.random.choice(len(self.trajs), p=self.sample_p)]
        T, K = len(t['actions']), self.K

        if T <= K:
            pad     = K - T
            states  = np.pad(t['states'],                  (0, pad)).astype(np.int64)
            actions = np.pad(t['actions'],                 (0, pad)).astype(np.int64)
            rtgs    = np.pad(t['rtg'] / self.rtg_scale,   (0, pad)).astype(np.float32)
            times   = np.pad(np.arange(T, dtype=np.int64),(0, pad))
            mask    = np.array([1.]*T + [0.]*pad, dtype=np.float32)
        else:
            s       = np.random.randint(0, T - K + 1)
            states  = t['states'] [s:s+K].astype(np.int64)
            actions = t['actions'][s:s+K].astype(np.int64)
            rtgs    = (t['rtg'][s:s+K] / self.rtg_scale).astype(np.float32)
            times   = np.arange(s, s+K, dtype=np.int64)
            mask    = np.ones(K, dtype=np.float32)

        return {
            'states'   : torch.from_numpy(states),
            'actions'  : torch.from_numpy(actions),
            'rtgs'     : torch.from_numpy(rtgs).unsqueeze(-1),
            'timesteps': torch.from_numpy(times),
            'mask'     : torch.from_numpy(mask),
        }


def worker_init_fn(wid: int):
    np.random.seed(HP['seed'] + wid)

## 5 · Model

In [10]:
class GPTBlock(nn.Module):
    """Pre-norm transformer block (GPT-style)."""

    def __init__(self, hidden_dim: int, n_heads: int, dropout: float):
        super().__init__()
        self.ln1  = nn.LayerNorm(hidden_dim)
        self.attn = nn.MultiheadAttention(
            hidden_dim, n_heads,
            dropout=dropout, batch_first=True,
        )
        self.ln2  = nn.LayerNorm(hidden_dim)
        self.ff   = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * hidden_dim, hidden_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        y = self.ln1(x)
        y, _ = self.attn(y, y, y, attn_mask=attn_mask, need_weights=False)
        x = x + y
        x = x + self.ff(self.ln2(x))
        return x


class DecisionTransformer(nn.Module):
    """
    Vanilla Decision Transformer for a discrete maze.

    Token order per timestep t: [RTG_t, state_t, action_t]
    Action is predicted from the hidden state at the *state* token position.
    Causal mask prevents action_t from attending to any future token.
    """

    def __init__(
        self,
        n_states   : int,
        n_actions  : int,
        hidden_dim : int,
        n_heads    : int,
        n_layers   : int,
        context_len: int,
        dropout    : float,
    ):
        super().__init__()
        self.hidden_dim  = hidden_dim
        self.context_len = context_len

        self.state_emb   = nn.Embedding(n_states,   hidden_dim)
        self.action_emb  = nn.Embedding(n_actions,  hidden_dim)
        self.rtg_emb     = nn.Linear(1, hidden_dim, bias=False)
        self.time_emb    = nn.Embedding(context_len, hidden_dim)
        self.input_ln    = nn.LayerNorm(hidden_dim)

        self.blocks      = nn.ModuleList(
            [GPTBlock(hidden_dim, n_heads, dropout) for _ in range(n_layers)]
        )
        self.final_ln    = nn.LayerNorm(hidden_dim)
        self.action_head = nn.Linear(hidden_dim, n_actions)

        max_seq = 3 * context_len
        causal  = torch.triu(torch.full((max_seq, max_seq), float('-inf')), diagonal=1)
        self.register_buffer('causal_mask', causal)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(
        self,
        states    : torch.Tensor,   # (B, T)  int64
        actions   : torch.Tensor,   # (B, T)  int64
        rtgs      : torch.Tensor,   # (B, T, 1) float32
        timesteps : torch.Tensor,   # (B, T)  int64
        mask      : Optional[torch.Tensor] = None,  # (B, T) float32
    ) -> torch.Tensor:              # (B, T, n_actions)
        B, T = states.shape

        t = self.time_emb(timesteps.clamp(0, self.context_len - 1))
        s = self.state_emb(states)  + t
        a = self.action_emb(actions)+ t
        r = self.rtg_emb(rtgs)      + t

        # Interleave: [RTG, state, action] × T  →  (B, 3T, H)
        x = torch.stack([r, s, a], dim=2).reshape(B, 3*T, self.hidden_dim)

        if mask is not None:
            triplet_mask = mask.unsqueeze(2).expand(B, T, 3).reshape(B, 3*T, 1)
            x = x * triplet_mask

        x      = self.input_ln(x)
        causal = self.causal_mask[:3*T, :3*T]
        for block in self.blocks:
            x = block(x, causal)
        x = self.final_ln(x)

        # Predict action from state-token positions (1, 4, 7, ...)
        return self.action_head(x[:, 1::3, :])

## 6 · Training Utilities (autograd / AMP)

In [11]:
def make_scheduler(optimizer, warmup_steps: int, total_steps: int):
    """Linear warmup → cosine decay to 10% of peak LR."""
    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * t))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class EarlyStopping:
    """
    Stops training when validation loss stops improving.

    Args:
        patience  : epochs to wait after last improvement
        min_delta : minimum change to count as improvement
    """

    def __init__(self, patience: int = 5, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float('inf')
        self.counter    = 0
        self.best_state : Optional[Dict] = None

    def step(self, val_loss: float, model: nn.Module) -> bool:
        """
        Call once per epoch.  Returns True when training should stop.
        Saves the best model state dict internally.
        """
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            # Deep-copy via state_dict so we don't hold a reference to live tensors
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            print(f'  ↩ Restored best weights (val_loss={self.best_loss:.4f})')


def run_epoch(
    model        : nn.Module,
    loader       : DataLoader,
    optimizer    : Optional[torch.optim.Optimizer],
    scheduler,
    scaler       : Optional[torch.cuda.amp.GradScaler],
    grad_clip    : float,
    grad_accum   : int,
    device       : torch.device,
    use_amp      : bool,
    train        : bool = True,
) -> float:
    """
    One epoch of training or validation.

    Uses torch.autograd for full gradient tracking (always on during train),
    and torch.cuda.amp for mixed-precision on CUDA.
    """
    model.train(train)
    ctx   = torch.enable_grad() if train else torch.no_grad()
    total = 0.0
    n     = 0

    with ctx:
        for step, batch in enumerate(loader):
            states    = batch['states']   .to(device, non_blocking=True)
            actions   = batch['actions']  .to(device, non_blocking=True)
            rtgs      = batch['rtgs']     .to(device, non_blocking=True)
            timesteps = batch['timesteps'].to(device, non_blocking=True)
            mask      = batch['mask']     .to(device, non_blocking=True)

            amp_ctx = torch.autocast(device_type=device.type, enabled=use_amp and device.type=='cuda')
            with amp_ctx:
                logits  = model(states, actions, rtgs, timesteps, mask)
                B, T, A = logits.shape
                valid   = mask.reshape(B*T).bool()
                lf      = logits.reshape(B*T, A)
                af      = actions.reshape(B*T)
                loss    = F.cross_entropy(lf[valid], af[valid]) if valid.any() else lf.sum() * 0.0
                loss    = loss / grad_accum

            if train:
                if scaler is not None:
                    scaler.scale(loss).backward()  # autograd through AMP
                else:
                    loss.backward()                # autograd, standard fp32

                if (step + 1) % grad_accum == 0:
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

            total += float(loss.item()) * grad_accum
            n     += 1

    return total / max(1, n)

## 7 · RTG Alignment Evaluation

In [12]:
@torch.no_grad()
def rollout_dt(
    model        : nn.Module,
    env          : MazeEnv,
    target_return: float,
    rtg_scale    : float,
    context_len  : int,
    max_steps    : int,
    device       : torch.device,
) -> Tuple[float, bool]:
    """Single greedy rollout conditioned on `target_return`. Returns (total_reward, reached_goal)."""
    model.eval()
    env.reset()
    pos   = env.start
    s_buf, a_buf, r_buf, t_buf = [], [], [], []
    rtg   = float(target_return)
    total = 0.0

    for step in range(max_steps):
        s_buf.append(env.enc(pos))
        a_buf.append(0)               # placeholder; filled after prediction
        r_buf.append(rtg / float(rtg_scale))
        t_buf.append(step)

        ctx = min(len(s_buf), context_len)
        states    = torch.tensor(s_buf[-ctx:], dtype=torch.long,    device=device).unsqueeze(0)
        actions   = torch.tensor(a_buf[-ctx:], dtype=torch.long,    device=device).unsqueeze(0)
        rtgs_t    = torch.tensor(r_buf[-ctx:], dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(-1)
        timesteps = torch.tensor(t_buf[-ctx:], dtype=torch.long,    device=device).unsqueeze(0)

        logits    = model(states, actions, rtgs_t, timesteps)[0, -1]
        valid     = env.valid_actions(pos)
        masked    = torch.full((N_ACTIONS,), float('-inf'), device=device)
        masked[valid] = logits[valid]
        action    = int(torch.argmax(masked).item())

        a_buf[-1] = action
        _, reward, done = env.step(action)
        pos    = env.pos
        total += reward
        rtg   -= reward          # DT RTG update: rtg_{t+1} = rtg_t - r_t

        if done:
            break

    return total, bool(pos == env.goal)


def check_rtg_alignment(target_rtg: float, net_return: float, delta: float) -> bool:
    """
    RTG alignment check.

    Returns True (SUCCESS) iff the realised return is within `delta` of the target.
    """
    return abs(target_rtg - net_return) <= delta


@torch.no_grad()
def evaluate_rtg_alignment(
    model      : nn.Module,
    env        : MazeEnv,
    rtg_scale  : float,
    context_len: int,
    max_steps  : int,
    device     : torch.device,
    target_rtgs: List[float],
    n_rollouts : int,
    delta      : float,
) -> Tuple[Dict, float]:
    """
    Evaluate RTG-alignment across multiple target returns.

    For each target RTG and each rollout we compute:
        aligned = |target_rtg - net_return| <= delta

    The overall alignment_score is the fraction of (target, rollout) pairs that pass.
    Additionally, the ordering score checks whether higher targets yield higher mean returns.

    Returns (results_dict, alignment_score).
    """
    results = {}
    total_trials   = 0
    total_successes= 0

    print()
    hdr = f"{'target_rtg':>12} {'mean_ret':>10} {'std_ret':>8} {'goal%':>7} {'align%':>8}"
    print(hdr)
    print('-' * len(hdr))

    for tgt in target_rtgs:
        returns, goals, aligned = [], [], []

        for _ in range(n_rollouts):
            ret, ok = rollout_dt(
                model, env, tgt, rtg_scale, context_len, max_steps, device
            )
            returns.append(ret)
            goals  .append(float(ok))
            success = check_rtg_alignment(tgt, ret, delta)
            aligned.append(float(success))

        mean_r     = float(np.mean(returns))
        std_r      = float(np.std(returns))
        goal_rate  = float(np.mean(goals))
        align_rate = float(np.mean(aligned))

        results[float(tgt)] = {
            'mean_return'  : mean_r,
            'goal_rate'    : goal_rate,
            'align_rate'   : align_rate,
            'returns'      : returns,
        }
        print(f"{tgt:>12.1f} {mean_r:>10.2f} {std_r:>8.2f} {goal_rate:>7.1%} {align_rate:>8.1%}")

        total_trials    += n_rollouts
        total_successes += sum(int(x) for x in aligned)

    # Fraction of all (target, rollout) pairs that pass the alignment check
    alignment_score = total_successes / max(1, total_trials)
    print(f'\nOverall alignment score: {alignment_score:.3f}  (delta={delta})')
    return results, alignment_score


def plot_alignment(results: Dict, target_rtgs: List[float], filename: str = 'alignment.png'):
    """Bar chart of per-target alignment rates + scatter of realised returns."""
    tgts      = sorted(results.keys())
    align_pct = [results[t]['align_rate'] * 100 for t in tgts]
    mean_rets = [results[t]['mean_return']        for t in tgts]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # ── Left: alignment rate ──
    ax1.bar([str(round(t,1)) for t in tgts], align_pct, color='steelblue', alpha=0.8)
    ax1.axhline(50, color='red', linestyle='--', linewidth=0.8, label='50%')
    ax1.set_xlabel('Target RTG')
    ax1.set_ylabel('Alignment rate (%)')
    ax1.set_title('RTG Alignment Rate per Target')
    ax1.legend()

    # ── Right: target vs mean realised ──
    ax2.plot(tgts, tgts,      'k--',  linewidth=0.8,  label='Perfect alignment')
    ax2.plot(tgts, mean_rets, 'o-',   color='coral', label='Mean realised return')
    ax2.set_xlabel('Target RTG')
    ax2.set_ylabel('Realised return')
    ax2.set_title('Target vs Realised Return')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(filename, dpi=120)
    plt.show()
    plt.close()
    print(f'Saved alignment plot → {filename}')

## 8 · Main Pipeline

In [13]:
# ── Step 1: Build maze ───────────────────────────────────────────────────────
print('=' * 70)
print(f"[1/5] Generating {HP['maze_rows']}×{HP['maze_cols']} maze")
print('=' * 70)

maze = make_maze(
    HP['maze_rows'], HP['maze_cols'],
    seed=HP['seed'],
    extra_opening_prob=HP['extra_opening_prob'],
)
env      = MazeEnv(maze, HP['maze_rows'], HP['maze_cols'])
dist     = bfs_dist_to_goal(env)
opt_len  = dist[env.start]
opt_ret  = -float(opt_len)
rtg_scale = float(opt_len) 
print(f'Shortest path: {opt_len} steps  |  optimal return: {opt_ret:.0f}')

visualise_maze(maze, env, filename='maze.png')

[1/5] Generating 200×200 maze
Shortest path: 414 steps  |  optimal return: -414
Saved maze image → maze.png


C:\Users\SYED NAVEED\AppData\Local\Temp\ipykernel_15768\3690075671.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ── Step 2: Collect trajectories ────────────────────────────────────────────
print('\n[2/5] Collecting trajectories')

all_trajs: List[Dict] = []
for i, eps in enumerate(HP['epsilons']):
    t0 = time.time()
    ts, attempts = collect_trajectories(
        env=env, dist_to_goal=dist,
        n_traj=HP['n_per_eps'], epsilon=eps,
        max_steps=HP['max_steps'],
        seed=HP['seed'] + i * 997,
        max_attempts_multiplier=HP['max_attempts_multiplier'],
    )
    all_trajs.extend(ts)
    avg_ret = np.mean([t['total_reward'] for t in ts]) if ts else float('nan')
    print(
        f'  eps={eps:.2f}  collected={len(ts)}/{HP["n_per_eps"]}'
        f'  attempts={attempts}  avg_return={avg_ret:.1f}'
        f'  ({time.time()-t0:.1f}s)'
    )
    if len(ts) < HP['n_per_eps']:
        print(f'   Warning: collected fewer than requested for eps={eps:.2f}')

if not all_trajs:
    raise RuntimeError('No successful trajectories. Increase max_steps or max_attempts_multiplier.')

with open(HP['traj_file'], 'wb') as f:
    pickle.dump(all_trajs, f)

rets = np.array([t['total_reward'] for t in all_trajs], dtype=np.float32)
print(f'\nSaved {len(all_trajs)} trajectories → {HP["traj_file"]}')
print(f'Return stats:  min={rets.min():.1f}  max={rets.max():.1f}  mean={rets.mean():.1f}')

In [ ]:
# ── Step 3: Dataset + DataLoaders ───────────────────────────────────────────
print('\n[3/5] Building train / val datasets')

rtg_scale = float(opt_len)   # normalise RTGs by shortest-path length
full_ds   = MazeTrajectoryDataset(
    trajectories=all_trajs,
    context_len=HP['context_len'],
    rtg_scale=rtg_scale,
    n_samples=HP['n_samples'],
)

val_size   = max(1, int(len(full_ds) * HP['val_fraction']))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(
    full_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(HP['seed']),
)

num_workers = 0
loader_kw   = dict(
    batch_size=HP['batch_size'], num_workers=num_workers,
    pin_memory=(DEVICE.type == 'cuda'),
    worker_init_fn=worker_init_fn,
    persistent_workers=(num_workers > 0),
)
train_loader = DataLoader(train_ds, shuffle=False, **loader_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kw)

print(f'Train samples : {train_size:,}  |  Val samples : {val_size:,}')
print(f'Train batches : {len(train_loader):,}  |  Val batches : {len(val_loader):,}')

In [ ]:
# ── Step 4: Build model ──────────────────────────────────────────────────────
print('\n[4/5] Building Decision Transformer')

model = DecisionTransformer(
    n_states   =N_STATES,
    n_actions  =N_ACTIONS,
    hidden_dim =HP['hidden_dim'],
    n_heads    =HP['n_heads'],
    n_layers   =HP['n_layers'],
    context_len=HP['context_len'],
    dropout    =HP['dropout'],
).to(DEVICE)

# Optional: graph-mode compilation (PyTorch >= 2.0, good for long runs)
if HP['use_compile'] and hasattr(torch, 'compile'):
    model = torch.compile(model)
    print('  torch.compile enabled')

n_params = sum(p.numel() for p in model.parameters())
print(f'  Parameters : {n_params:,}')

In [ ]:
# ── Step 5: Train with autograd + AMP + early stopping ──────────────────────
print('\n[5/5] Training')

optimizer   = torch.optim.AdamW(
    model.parameters(),
    lr=HP['lr'], weight_decay=HP['weight_decay'],
)
total_steps = HP['epochs'] * (len(train_loader) // HP['grad_accum_steps'])
scheduler   = make_scheduler(optimizer, HP['warmup_steps'], total_steps)

# AMP scaler — no-op on CPU
use_amp = HP['use_amp'] and DEVICE.type == 'cuda'
scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

stopper    = EarlyStopping(patience=HP['patience'], min_delta=HP['min_delta'])
train_hist = []
val_hist   = []

print(f'AMP={use_amp}  |  GradAccum={HP["grad_accum_steps"]}  |  Patience={HP["patience"]}')
print()

header = f"{'Epoch':>6}  {'Train':>8}  {'Val':>8}  {'LR':>10}  {'Pat':>5}"
print(header)
print('-' * len(header))

for epoch in range(1, HP['epochs'] + 1):
    t0 = time.time()

    train_loss = run_epoch(
        model, train_loader, optimizer, scheduler, scaler,
        HP['grad_clip'], HP['grad_accum_steps'], DEVICE, use_amp, train=True,
    )
    val_loss = run_epoch(
        model, val_loader, None, None, None,
        HP['grad_clip'], HP['grad_accum_steps'], DEVICE, use_amp, train=False,
    )

    train_hist.append(train_loss)
    val_hist  .append(val_loss)
    cur_lr = scheduler.get_last_lr()[0] if hasattr(scheduler, 'get_last_lr') else HP['lr']

    print(
        f"{epoch:>6}  {train_loss:>8.4f}  {val_loss:>8.4f}"
        f"  {cur_lr:>10.2e}  {stopper.counter:>5}"
        f"  [{time.time()-t0:.1f}s]"
    )

    if stopper.step(val_loss, model):
        print(f'\n⏹ Early stopping triggered at epoch {epoch} (patience={HP["patience"]})')
        break

stopper.restore_best(model)

# Save checkpoint
torch.save({
    'model'    : model.state_dict(),
    'rtg_scale': rtg_scale,
    'opt_len'  : opt_len,
    'hp'       : HP,
    'train_hist': train_hist,
    'val_hist'  : val_hist,
}, HP['ckpt_file'])
print(f'\nCheckpoint saved → {HP["ckpt_file"]}')

In [ ]:
# ── Training loss curve ──────────────────────────────────────────────────────
epochs_ran = range(1, len(train_hist) + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs_ran, train_hist, label='Train loss', color='royalblue')
plt.plot(epochs_ran, val_hist,   label='Val loss',   color='coral')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Decision Transformer Training Curve')
plt.legend()
plt.tight_layout()
plt.savefig('training_curve.png', dpi=120)
plt.show()
plt.close()
print('Saved training_curve.png')

In [12]:
# ── RTG Alignment Evaluation ─────────────────────────────────────────────────
print('\nEvaluating RTG alignment...')
quantiles = np.quantile(rets, np.linspace(0, 1, 100))
target_rtgs = [float(q) for q in quantiles]
print(f'Target RTGs: {[round(t,1) for t in target_rtgs]}')
print(f'Delta (tolerance): {HP["rtg_delta"]}')

results, score = evaluate_rtg_alignment(
    model=model,
    env=env,
    rtg_scale=rtg_scale,
    context_len=HP['context_len'],
    max_steps=HP['max_steps'],
    device=DEVICE,
    target_rtgs=target_rtgs,
    n_rollouts=HP['eval_rollouts'],
    delta=HP['rtg_delta'],
)

out = {
    'target_rtgs'     : target_rtgs,
    'results'         : results,
    'alignment_score' : score,
    'delta'           : HP['rtg_delta'],
}
with open(HP['eval_file'], 'wb') as f:
    pickle.dump(out, f)
print(f'\nEvaluation saved → {HP["eval_file"]}')

plot_alignment(results, target_rtgs, filename=f"alignment_{HP['context_len']}_test.png")


Evaluating RTG alignment...


NameError: name 'rets' is not defined

In [14]:
import torch
import pickle
import numpy as np

# ── Load Model ────────────────────────────────────────────────────────────────
model = DecisionTransformer(
    n_states   =N_STATES,
    n_actions  =N_ACTIONS,
    hidden_dim =HP['hidden_dim'],
    n_heads    =HP['n_heads'],
    n_layers   =HP['n_layers'],
    context_len=HP['context_len'],
    dropout    =HP['dropout'],
).to(DEVICE)

checkpoint = torch.load("dt_checkpoint.pt", map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
model.to(DEVICE)
model.eval()

print(f"Loaded model")

with open(HP["traj_file"], "rb") as f:
    all_trajs = pickle.load(f)

rets = np.array([t['total_reward'] for t in all_trajs], dtype=np.float32)

print(f'Loaded {len(all_trajs)} trajectories → {HP["traj_file"]}')
print(f'Return stats:  min={rets.min():.1f}  max={rets.max():.1f}  mean={rets.mean():.1f}')


# ── RTG Alignment Evaluation ─────────────────────────────────────────────────
print('\nEvaluating RTG alignment...')

quantiles = np.quantile(rets, np.linspace(0, 1, 25))
# target_rtgs = [float(q) for q in quantiles]
target_rtgs=[-6000, -5000, -4000, -2000,-2100,-2200,-1900,-1700,-2210,-1910,-1710,-300,-404]

print(f'Target RTGs: {[round(t,1) for t in target_rtgs]}')
print(f'Delta (tolerance): {HP["rtg_delta"]}')

results, score = evaluate_rtg_alignment(
    model=model,
    env=env,
    rtg_scale=rtg_scale,
    context_len=HP['context_len'],
    max_steps=HP['max_steps'],
    device=DEVICE,
    target_rtgs=target_rtgs,
    n_rollouts=HP['eval_rollouts'],
    delta=HP['rtg_delta'],
)

# ── Save Results ─────────────────────────────────────────────────────────────
out = {
    'target_rtgs'     : target_rtgs,
    'results'         : results,
    'alignment_score' : score,
    'delta'           : HP['rtg_delta'],
}

with open(HP['eval_file'], 'wb') as f:
    pickle.dump(out, f)

print(f'\nEvaluation saved → {HP["eval_file"]}')

# ── Plot ─────────────────────────────────────────────────────────────────────
plot_alignment(results, target_rtgs, filename=f"alignment_{HP['context_len']}_test_2.png")

C:\Users\SYED NAVEED\AppData\Local\Temp\ipykernel_15768\3949504324.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("dt_checkpoint.pt", map_locat

Loaded model
Loaded 500 trajectories → trajectories.pkl
Return stats:  min=-2996.0  max=-414.0  mean=-1149.2

Evaluating RTG alignment...
Target RTGs: [-6000, -5000, -4000, -2000, -2100, -2200, -1900, -1700, -2210, -1910, -1710, -300, -404]
Delta (tolerance): 12.0

  target_rtg   mean_ret  std_ret   goal%   align%
-------------------------------------------------
     -6000.0   -3000.00     0.00    0.0%     0.0%
     -5000.0   -3000.00     0.00    0.0%     0.0%
     -4000.0   -3000.00     0.00    0.0%     0.0%
     -2000.0   -2006.00     0.00  100.0%   100.0%
     -2100.0   -2182.00     0.00  100.0%     0.0%
     -2200.0   -2246.00     0.00  100.0%     0.0%
     -1900.0   -1936.00     0.00  100.0%     0.0%
     -1700.0   -1840.00     0.00  100.0%     0.0%
     -2210.0   -2218.00     0.00  100.0%   100.0%
     -1910.0   -1968.00     0.00  100.0%     0.0%
     -1710.0   -1728.00     0.00  100.0%     0.0%
      -300.0    -414.00     0.00  100.0%     0.0%
      -404.0    -414.00     0.00  

C:\Users\SYED NAVEED\AppData\Local\Temp\ipykernel_15768\1055509738.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('=' * 70)
print('SUMMARY')
print('=' * 70)
print(f'  Maze             : {HP["maze_rows"]}×{HP["maze_cols"]}')
print(f'  Optimal return   : {opt_ret:.0f}')
print(f'  Trajectories     : {len(all_trajs):,}')
print(f'  Model params     : {n_params:,}')
print(f'  Epochs run       : {len(train_hist)}')
print(f'  Best val loss    : {stopper.best_loss:.4f}')
print(f'  RTG delta        : {HP["rtg_delta"]}')
print(f'  Alignment score  : {score:.3f}  (fraction of rollouts within delta)')
print()
for tgt, row in sorted(results.items()):
    status = 'PASS' if row['align_rate'] >= 0.5 else 'FAIL'
    print(
        f'  target={tgt:>8.1f}  mean_ret={row["mean_return"]:>7.2f}'
        f'  align={row["align_rate"]:.1%}  [{status}]'
    )
print('=' * 70)